# Beca 18 RAG Chatbot
**Fuente:** Resolución Directoral Ejecutiva N.° 033-2026-MINEDU/VMGI-PRONABEC  
**Pipeline:** PDF → extracción → chunks → embeddings → ChromaDB → búsqueda semántica → respuesta fundamentada

## Step 0 — Setup

Instalamos las dependencias necesarias y cargamos la API key de Gemini desde un archivo `.env` usando `python-dotenv`. La clave **nunca** se escribe directamente en el notebook.

In [1]:
# Install dependencies (run once)
# !pip install pypdf==4.3.1 tiktoken==0.7.0 langchain-text-splitters==0.2.4 \
#              google-genai==0.8.0 chromadb==0.5.5 ipywidgets==8.1.5 \
#              tqdm==4.66.4 python-dotenv==1.0.1

In [2]:
import os
import re
import time
import math
import importlib
from pathlib import Path

import pypdf
import tiktoken
import chromadb
import google.genai as genai
import ipywidgets as widgets
from tqdm.auto import tqdm
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load API key from .env
load_dotenv(dotenv_path=Path("..") / ".env")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "GEMINI_API_KEY not found. Create a .env file with GEMINI_API_KEY=your_key_here"

# Print package versions
for pkg in ["pypdf", "tiktoken", "chromadb", "google.genai", "ipywidgets"]:
    mod = importlib.import_module(pkg)
    version = getattr(mod, "__version__", "n/a")
    print(f"{pkg}: {version}")

print("\nEnvironment ready. API key loaded.")

pypdf: 4.3.1
tiktoken: 0.12.0
chromadb: 1.5.8
google.genai: 1.16.0
ipywidgets: 8.1.5

Environment ready. API key loaded.


## Step 1 — PDF Text Extraction

Extraemos el texto página por página con `pypdf`. Insertamos un marcador `[PAGE N]` al inicio de cada página para poder citar la fuente después. Aplicamos limpieza ligera: colapsamos espacios múltiples y eliminamos saltos de línea aislados.

In [3]:
PDF_PATH = Path("..") / "data" / "beca18_reglamento.pdf"

def clean_page(text: str) -> str:
    """Light cleaning: collapse spaces and join orphan line breaks."""
    text = re.sub(r" {2,}", " ", text)          # multiple spaces → one
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text) # isolated newlines → space
    text = re.sub(r"\n{3,}", "\n\n", text)       # 3+ newlines → 2
    return text.strip()

pages_text = []
with open(PDF_PATH, "rb") as f:
    reader = pypdf.PdfReader(f)
    for i, page in enumerate(reader.pages, start=1):
        raw = page.extract_text() or ""
        cleaned = clean_page(raw)
        pages_text.append(f"[PAGE {i}]\n{cleaned}")

full_text = "\n\n".join(pages_text)

print(f"Pages extracted : {len(pages_text)}")
print(f"Total characters: {len(full_text):,}")
print(f"Total words     : {len(full_text.split()):,}")
print("\n--- First 500 characters ---")
print(full_text[:500])

Pages extracted : 138
Total characters: 374,961
Total words     : 56,074

--- First 500 characters ---
[PAGE 1]
Resolución Directoral Ejecutiva  Nº 033-2026 -MINEDU/VMGI -PRONABEC     Lima, 24 de febrero de 2026    VISTOS:    El Informe N° 451 -2026 -MINEDU/VMGI -PRONABEC -DIBEC -SES, suscrito por  la Dirección de Gestión de Becas y la Dirección de Acompañamiento Socioemocional y  Bienestar; el Informe N° 042-2026 -MINEDU/VMGI -PRONABEC -OPP de la Oficina de  Planeamiento y Presupuesto; el Informe N ° 048-2026 -MINEDU/VMGI -PRONABEC -OAJ  de la Oficina de Asesoría Jurídica, y;    CONSIDERANDO:   


## Step 2 — Tokenización y Justificación del Chunking

Contamos los tokens con `tiktoken` (encoding `cl100k_base`, estándar de OpenAI y compatible con la mayoría de modelos modernos).

In [4]:
enc = tiktoken.get_encoding("cl100k_base")
total_tokens = len(enc.encode(full_text))
print(f"Total tokens (cl100k_base): {total_tokens:,}")

Total tokens (cl100k_base): 108,903


### Justificación del tamaño de chunk

El modelo de embeddings `gemini-embedding-001` acepta hasta **8 192 tokens** por fragmento. Sin embargo, fragmentos muy grandes:

- **Diluyen la señal semántica**: al mezclar varios temas en un solo vector, la similitud coseno con la pregunta baja, y el retriever recupera chunks menos precisos.
- **Encarecen el contexto del LLM**: pasar k=5 chunks de 8 000 tokens cada uno consume ~40 000 tokens de contexto.

Un **chunk_size de 400 tokens** (~300 palabras) cubre típicamente uno o dos artículos del reglamento — unidad lógica mínima con sentido completo. El **overlap de 60 tokens** (~15 % del chunk) garantiza que ninguna oración quede cortada entre dos fragmentos, preservando la continuidad de las condiciones redactadas en párrafos que cruzan el límite de corte. Este balance maximiza la precisión del retriever sin desperdiciar contexto del LLM.

In [5]:
CHUNK_SIZE    = 400
CHUNK_OVERLAP = 60

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " "],
    length_function=len,
)

base_metadata = {
    "document": "Resolución Directoral Ejecutiva N.° 033-2026-MINEDU/VMGI-PRONABEC",
    "topic"   : "Beca 18 Reglamento",
    "language": "es",
}

raw_chunks = splitter.create_documents(
    texts=[full_text],
    metadatas=[base_metadata],
)

# Attach page number extracted from [PAGE N] markers inside each chunk
def extract_page(text: str) -> str:
    match = re.search(r"\[PAGE (\d+)\]", text)
    return match.group(1) if match else "unknown"

chunks = []
for doc in raw_chunks:
    doc.metadata["page"] = extract_page(doc.page_content)
    chunks.append(doc)

avg_len = sum(len(c.page_content) for c in chunks) / len(chunks)
print(f"Total chunks   : {len(chunks)}")
print(f"Avg chunk length: {avg_len:.0f} characters")
print("\n--- Sample chunk ---")
print(chunks[5].page_content[:300])
print("Metadata:", chunks[5].metadata)

Total chunks   : 1485
Avg chunk length: 264 characters

--- Sample chunk ---
económicos  y alto rendimiento académico, así como su permanencia y culmina ción;    Que, el literal b) del artículo 5 del Reglamento de la Ley N° 29837, Ley que crea  el PRONABEC, aprobado por Decreto Supremo N° 018 -2020 -MINEDU (en adelante, el  Reglamento), establece que las Bases son las reglas
Metadata: {'document': 'Resolución Directoral Ejecutiva N.° 033-2026-MINEDU/VMGI-PRONABEC', 'topic': 'Beca 18 Reglamento', 'language': 'es', 'page': 'unknown'}


## Step 3 — Embeddings

Usamos `gemini-embedding-001` (768 dimensiones). La distinción de `task_type` es importante:
- **RETRIEVAL_DOCUMENT**: optimizado para textos que van a ser indexados (los chunks del PDF).
- **RETRIEVAL_QUERY**: optimizado para la pregunta del usuario en tiempo de búsqueda.

Añadimos **exponential backoff** para manejar el límite del tier gratuito (~60 RPM).

In [6]:
client_genai = genai.Client(api_key=GEMINI_API_KEY)
EMBEDDING_MODEL = "gemini-embedding-001"

def _embed_with_retry(texts: list[str], task_type: str, max_retries: int = 6) -> list[list[float]]:
    """Call Gemini embedding API with exponential backoff on rate-limit errors."""
    for attempt in range(max_retries):
        try:
            response = client_genai.models.embed_content(
                model=EMBEDDING_MODEL,
                contents=texts,
                config=genai.types.EmbedContentConfig(task_type=task_type),
            )
            return [e.values for e in response.embeddings]
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            wait = (2 ** attempt) + 1
            print(f"Rate limit hit, retrying in {wait}s... (attempt {attempt+1})")
            time.sleep(wait)

def embed_documents(texts: list[str], batch_size: int = 10) -> list[list[float]]:
    """Embed a list of document texts in batches (RETRIEVAL_DOCUMENT task type)."""
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding docs"):
        batch = texts[i : i + batch_size]
        all_embeddings.extend(_embed_with_retry(batch, task_type="RETRIEVAL_DOCUMENT"))
        time.sleep(1)  # stay under free-tier RPM
    return all_embeddings

def embed_query(text: str) -> list[float]:
    """Embed a single user query (RETRIEVAL_QUERY task type)."""
    return _embed_with_retry([text], task_type="RETRIEVAL_QUERY")[0]

print(f"Embedding functions ready. Model: {EMBEDDING_MODEL}")
# Quick smoke test
test_vec = embed_query("¿Qué es Beca 18?")
print(f"Query embedding dimensions: {len(test_vec)}")

Embedding functions ready. Model: gemini-embedding-001
Rate limit hit, retrying in 2s... (attempt 1)
Query embedding dimensions: 3072


## Step 4 — Vector Database (ChromaDB)

Creamos una colección **persistente** en disco con distancia **coseno**. El indexado es **idempotente**: si la colección ya tiene documentos, saltamos el embedding y reutilizamos lo que hay.

In [7]:
CHROMA_PATH      = "../chroma_db_beca18"
COLLECTION_NAME  = "beca18_reglamento"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

existing_count = collection.count()
print(f"Documents already in collection: {existing_count}")

if existing_count == 0:
    print("Collection empty — starting embedding and indexing...")
    texts     = [c.page_content for c in chunks]
    metadatas = [c.metadata     for c in chunks]
    ids       = [f"chunk_{i}"   for i in range(len(chunks))]

    embeddings = embed_documents(texts)

    collection.add(
        documents=texts,
        embeddings=embeddings,
        metadatas=metadatas,
        ids=ids,
    )
    print(f"Indexed {collection.count()} documents.")
else:
    print("Collection already populated — skipping embedding step.")

print(f"\nTotal documents stored: {collection.count()}")

Documents already in collection: 0
Collection empty — starting embedding and indexing...


Embedding docs:   0%|          | 0/149 [00:00<?, ?it/s]

Rate limit hit, retrying in 2s... (attempt 1)
Rate limit hit, retrying in 3s... (attempt 2)
Rate limit hit, retrying in 5s... (attempt 3)
Rate limit hit, retrying in 9s... (attempt 4)
Rate limit hit, retrying in 17s... (attempt 5)
Rate limit hit, retrying in 2s... (attempt 1)
Rate limit hit, retrying in 2s... (attempt 1)
Rate limit hit, retrying in 3s... (attempt 2)
Rate limit hit, retrying in 5s... (attempt 3)
Rate limit hit, retrying in 9s... (attempt 4)
Rate limit hit, retrying in 17s... (attempt 5)
Rate limit hit, retrying in 2s... (attempt 1)
Rate limit hit, retrying in 2s... (attempt 1)
Rate limit hit, retrying in 2s... (attempt 1)
Rate limit hit, retrying in 2s... (attempt 1)
Rate limit hit, retrying in 2s... (attempt 1)
Rate limit hit, retrying in 3s... (attempt 2)
Rate limit hit, retrying in 5s... (attempt 3)
Rate limit hit, retrying in 9s... (attempt 4)
Rate limit hit, retrying in 17s... (attempt 5)
Rate limit hit, retrying in 2s... (attempt 1)
Rate limit hit, retrying in 3s.

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-1.0\nPlease retry in 48.004421619s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerDayPerUserPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-e

## Step 5 — Semantic Search

La función `semantic_search` convierte la pregunta en un embedding con `embed_query` y luego consulta ChromaDB para obtener los `k` chunks más cercanos. Retorna una lista de dicts con `text`, `metadata` y `distance`.

In [ ]:
def semantic_search(question: str, k: int = 5) -> list[dict]:
    """Return top-k chunks most semantically similar to the question."""
    query_vec = embed_query(question)
    results   = collection.query(
        query_embeddings=[query_vec],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    output = []
    for text, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        output.append({"text": text, "metadata": meta, "distance": dist})
    return output

# Test with a sample question — print top-3
sample_results = semantic_search("¿Cuáles son los requisitos de elegibilidad para Beca 18?", k=3)
print(f"Top-3 results for sample question:\n")
for i, r in enumerate(sample_results, 1):
    print(f"[{i}] Distance: {r['distance']:.4f} | Page: {r['metadata'].get('page')}")
    print(r['text'][:200])
    print()

## Step 6 — Grounded Generation

Usamos `gemini-2.5-flash`. El system prompt instruye al modelo a:
1. Responder **exclusivamente** desde el contexto recuperado.
2. **Citar el número de página** cuando esté disponible.
3. Responder con una frase estándar si el contexto es insuficiente — sin alucinar.

In [ ]:
LLM_MODEL = "gemini-2.5-flash"

SYSTEM_PROMPT = """Eres un asistente especializado en el reglamento de Beca 18 (PRONABEC, Perú).
Responde ÚNICAMENTE basándote en los fragmentos del documento oficial que se te proporcionan como contexto.
Cuando cites información, indica el número de página entre paréntesis, por ejemplo: (Página 5).
Si el contexto proporcionado no contiene información suficiente para responder la pregunta, responde exactamente:
"El documento no contiene información sobre este tema."
Nunca uses conocimiento externo al documento. Nunca inventes datos, cifras ni condiciones."""

def build_context_block(results: list[dict]) -> str:
    lines = []
    for i, r in enumerate(results, 1):
        page = r["metadata"].get("page", "?")
        lines.append(f"--- Fragmento {i} (Página {page}) ---\n{r['text']}")
    return "\n\n".join(lines)

def answer_with_context(question: str, k: int = 5) -> dict:
    """Retrieve relevant chunks and generate a grounded answer."""
    results      = semantic_search(question, k=k)
    context_block = build_context_block(results)
    user_message = f"CONTEXTO DEL DOCUMENTO:\n{context_block}\n\nPREGUNTA: {question}"

    response = client_genai.models.generate_content(
        model=LLM_MODEL,
        contents=user_message,
        config=genai.types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0.0,
        ),
    )
    return {"answer": response.text, "sources": results}

print("answer_with_context ready.")

### Pruebas: 5 preguntas temáticas + 1 fuera de tema

In [ ]:
test_questions = [
    # On-topic
    "¿Cuáles son los requisitos de elegibilidad para postular a Beca 18?",
    "¿Cuáles son las modalidades de becas que ofrece Beca 18?",
    "¿Cuál es el monto mensual de la subvención económica de Beca 18?",
    "¿Cuáles son las obligaciones del becario durante su formación?",
    "¿Bajo qué condiciones se puede perder la beca?",
    # Off-topic
    "¿Cuál es la receta del ceviche peruano?",
]

for q in test_questions:
    print(f"\n{'='*70}")
    print(f"PREGUNTA: {q}")
    print("-"*70)
    result = answer_with_context(q, k=5)
    print(f"RESPUESTA:\n{result['answer']}")

## Step 7 — Interactive Chat Interface

Interfaz con `ipywidgets` que incluye:
- Caja de texto para la pregunta.
- Botones **Preguntar** y **Limpiar**.
- Slider para controlar `k` (chunks recuperados).
- Área de respuesta + acordeón expandible con los fragmentos fuente, página y distancia.

In [ ]:
from IPython.display import display, HTML

# ── Widgets ──────────────────────────────────────────────────────────────────
question_input = widgets.Textarea(
    placeholder="Escribe tu pregunta sobre Beca 18...",
    layout=widgets.Layout(width="100%", height="70px"),
)

k_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description="k (chunks):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="350px"),
)

ask_button   = widgets.Button(description="Preguntar", button_style="primary", icon="search")
clear_button = widgets.Button(description="Limpiar",   button_style="warning", icon="trash")
status_label = widgets.Label(value="")

answer_out  = widgets.Output()
sources_out = widgets.Output()

accordion = widgets.Accordion(children=[sources_out])
accordion.set_title(0, "Fragmentos fuente")
accordion.selected_index = None  # collapsed by default

# ── Callbacks ────────────────────────────────────────────────────────────────
def on_ask(b):
    question = question_input.value.strip()
    if not question:
        status_label.value = "Escribe una pregunta primero."
        return

    status_label.value = "Buscando y generando respuesta..."
    ask_button.disabled = True

    answer_out.clear_output()
    sources_out.clear_output()

    try:
        result = answer_with_context(question, k=k_slider.value)

        with answer_out:
            display(HTML(f"""
                <div style='border:1px solid #ccc; border-radius:6px; padding:12px;
                            background:#f9f9f9; font-size:14px; line-height:1.6'>
                    <b>Respuesta:</b><br>{result['answer'].replace(chr(10), '<br>')}
                </div>
            """))

        with sources_out:
            for i, src in enumerate(result["sources"], 1):
                page = src["metadata"].get("page", "?")
                dist = src["distance"]
                snippet = src["text"][:350].replace("<", "&lt;").replace(">", "&gt;")
                display(HTML(f"""
                    <div style='border:1px solid #ddd; border-radius:4px; padding:8px;
                                margin-bottom:8px; font-size:12px; background:#fff'>
                        <b>Fragmento {i}</b> &nbsp;|&nbsp; Página: <b>{page}</b>
                        &nbsp;|&nbsp; Distancia: <b>{dist:.4f}</b>
                        <hr style='margin:6px 0'>
                        {snippet}{'...' if len(src['text']) > 350 else ''}
                    </div>
                """))

        status_label.value = f"Listo. {len(result['sources'])} fragmentos recuperados."
        accordion.selected_index = None

    except Exception as e:
        with answer_out:
            display(HTML(f"<p style='color:red'>Error: {e}</p>"))
        status_label.value = "Error al procesar la pregunta."
    finally:
        ask_button.disabled = False

def on_clear(b):
    question_input.value = ""
    answer_out.clear_output()
    sources_out.clear_output()
    status_label.value = ""
    accordion.selected_index = None

ask_button.on_click(on_ask)
clear_button.on_click(on_clear)

# ── Layout ───────────────────────────────────────────────────────────────────
controls = widgets.HBox([ask_button, clear_button, k_slider])

ui = widgets.VBox([
    widgets.HTML("<h3>Chatbot Beca 18 — Preguntas sobre el Reglamento</h3>"),
    question_input,
    controls,
    status_label,
    answer_out,
    accordion,
])

display(ui)